
# NLP Unstructured Insights: Four-Track Modeling Notebook

This notebook upgrades the project into four serious modeling labs under one consistent validation and ranking framework:

1. LazyPredict Discovery Lab
2. Manual Engineering Lab
3. FLAML Optimization Lab
4. PyCaret Experiment Lab



## 1) Business Problem and Success Criteria

**Business problem**

We need to convert noisy customer reviews into actionable operational intelligence for:

- support triage,
- product quality feedback,
- service and packaging issue discovery.

**Primary ML task**

Multiclass classification of review text into operational action categories.

**Success criteria**

- Primary: **macro F1** (class-balanced performance)
- Secondary: **weighted F1**
- Tertiary: **critical-issue recall** on `quality_issue` + `shipping_service_issue`
- Confidence quality: calibration-oriented confidence metric
- Operational: clear recommendation and deployable artifact


In [ ]:

from __future__ import annotations

import json
import subprocess
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, train_test_split

from src.text_prep import PreprocessingConfig, prepare_reviews_frame, top_terms_by_category
from src.insights_pipeline import (
    MANUAL_FAMILY_DISPLAY,
    classification_metrics_bundle,
    evaluate_manual_family,
    load_inference_bundle,
    make_leaderboard_row,
    optimize_priority_threshold,
    predict_with_inference_bundle,
    qualitative_top20_review,
    rank_leaderboard,
    rerun_candidates_with_multiple_seeds,
    run_flaml_optimization,
    run_lazypredict_discovery,
    run_pycaret_experiment,
    save_inference_bundle,
    save_leaderboard,
    select_top_eligible_from_lazypredict,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

PROJECT_NAME = "nlp-unstructured-insights"
RANDOM_STATE = 42
PRIMARY_METRIC = "macro_f1"
SECONDARY_METRIC = "weighted_f1"
TERTIARY_METRIC = "critical_issue_recall"

ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts"
for p in [DATA_RAW / "amazon", DATA_RAW / "yelp", DATA_PROCESSED, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 180)
sns.set_theme(style="whitegrid")

print(f"Project root: {ROOT}")
print(f"Artifact directory: {ARTIFACTS}")



## 2) Dataset Access and Data Dictionary

Primary dataset remains **Amazon Fine Food Reviews**.

Source: <https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews>


In [ ]:

amazon_csv = DATA_RAW / "amazon" / "Reviews.csv"
if not amazon_csv.exists():
    print("Dataset not found locally. Attempting Kaggle download...")
    try:
        subprocess.run(
            [
                "uv",
                "run",
                "kaggle",
                "datasets",
                "download",
                "-d",
                "snap/amazon-fine-food-reviews",
                "-p",
                str(DATA_RAW / "amazon"),
                "--unzip",
            ],
            check=True,
        )
    except Exception as exc:
        print("Automatic download failed. Run manually:")
        print("uv run kaggle datasets download -d snap/amazon-fine-food-reviews -p data/raw/amazon --unzip")
        raise

raw_df = pd.read_csv(
    amazon_csv,
    usecols=["Id", "ProductId", "UserId", "ProfileName", "HelpfulnessNumerator", "HelpfulnessDenominator", "Score", "Time", "Summary", "Text"],
    low_memory=False,
)

print(f"Raw rows: {len(raw_df):,}")
display(raw_df.head(3))

data_dictionary = pd.DataFrame(
    {
        "column": [
            "Id",
            "ProductId",
            "UserId",
            "Score",
            "Summary",
            "Text",
            "Time",
            "HelpfulnessNumerator",
            "HelpfulnessDenominator",
        ],
        "meaning": [
            "Review row id",
            "Product identifier",
            "User identifier",
            "Star score (1-5)",
            "Short review summary",
            "Full review text",
            "Unix timestamp",
            "Helpful votes numerator",
            "Helpful votes denominator",
        ],
        "used_in_model": ["No", "No", "No", "No (leakage risk)", "No", "Yes", "No", "No", "No"],
    }
)
data_dictionary



## 3) Data Cleaning and Leakage Checks

We keep text-only modeling to avoid label leakage from rating or metadata proxies.


In [ ]:

clean_df = prepare_reviews_frame(
    raw_df,
    PreprocessingConfig(
        text_col="Text",
        score_col="Score",
        sample_size=14000,
        random_state=RANDOM_STATE,
    ),
)

# Persist cleaned sample for reproducibility
clean_df.to_csv(DATA_PROCESSED / "amazon_reviews_clean_sample.csv", index=False)

duplicate_text_ratio = raw_df.duplicated(subset=["Text"]).mean()
empty_clean_ratio = (clean_df["clean_text"].str.len() == 0).mean()
label_distribution = clean_df["action_category"].value_counts(normalize=True).rename("share")

leakage_report = pd.DataFrame(
    {
        "check": [
            "Duplicate raw text ratio",
            "Empty cleaned text ratio",
            "Uses `Score` as model feature",
            "Uses user/product ids as model feature",
        ],
        "value": [
            round(float(duplicate_text_ratio), 4),
            round(float(empty_clean_ratio), 4),
            "No",
            "No",
        ],
    }
)

print(f"Prepared rows: {len(clean_df):,}")
display(leakage_report)
display(label_distribution.head(12))

print("Label vs score distribution snapshot (for leakage awareness, not as feature):")
display(pd.crosstab(clean_df["Score"], clean_df["action_category"], normalize="index").round(3).head())



## 4) Feature Engineering

Feature set for all tracks:

- TF-IDF word n-grams `(1,2)` with vocabulary trimming
- Dense SVD projection for algorithms needing dense matrices (LazyPredict, PyCaret)
- Raw sparse TF-IDF matrix for manual and FLAML tracks


In [ ]:

TARGET_COL = "action_category"
min_class_size = 80

label_counts = clean_df[TARGET_COL].value_counts()
eligible_labels = label_counts[label_counts >= min_class_size].index
model_df = clean_df[clean_df[TARGET_COL].isin(eligible_labels)].copy().reset_index(drop=True)

vectorizer_params = {
    "max_features": 12000,
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.92,
    "sublinear_tf": True,
}

print(f"Modeling rows after class-size filter: {len(model_df):,}")
print(f"Classes used ({len(eligible_labels)}): {list(eligible_labels)}")

display(top_terms_by_category(model_df).head(8))



## 5) Validation Strategy

- Stratified train/holdout split for fair class representation
- Cross-validation inside each serious track where applicable
- Primary ranking metric fixed to macro F1 across tracks


In [ ]:

train_df, valid_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=model_df[TARGET_COL],
)

train_text = train_df["clean_text"]
valid_text = valid_df["clean_text"]
y_train = train_df[TARGET_COL]
y_valid = valid_df[TARGET_COL]

# Fit transformers on train only to avoid leakage.
vectorizer = TfidfVectorizer(**vectorizer_params)
X_train_sparse = vectorizer.fit_transform(train_text)
X_valid_sparse = vectorizer.transform(valid_text)

n_svd = min(220, X_train_sparse.shape[1] - 1)
svd = TruncatedSVD(n_components=n_svd, random_state=RANDOM_STATE)
X_train_dense = svd.fit_transform(X_train_sparse)
X_valid_dense = svd.transform(X_valid_sparse)
dense_feature_names = [f"svd_{i:03d}" for i in range(X_train_dense.shape[1])]

overlap_count = len(set(train_text).intersection(set(valid_text)))
cv_strategy = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

print("Validation setup")
print(f"- Train rows: {len(train_df):,}")
print(f"- Holdout rows: {len(valid_df):,}")
print(f"- Train/holdout text overlap: {overlap_count}")
print(f"- Sparse shape train/holdout: {X_train_sparse.shape} / {X_valid_sparse.shape}")
print(f"- Dense shape train/holdout: {X_train_dense.shape} / {X_valid_dense.shape}")



## 6) LazyPredict Discovery Lab

Purpose: rapid model-family discovery on a consistent feature matrix and holdout split.


In [ ]:

lazy_models_df, lazy_rows_df = run_lazypredict_discovery(
    x_train=X_train_dense,
    y_train=y_train,
    x_valid=X_valid_dense,
    y_valid=y_valid,
    project_name=PROJECT_NAME,
    max_models=22,
    timeout_sec=420,
    top_n_rows=20,
)

if lazy_models_df.empty:
    print("LazyPredict unavailable or failed in this environment.")
else:
    lazy_models_df.to_csv(ARTIFACTS / "lazypredict_discovery_full.csv", index=False)
    lazy_rows_df.to_csv(ARTIFACTS / "lazypredict_discovery_leaderboard_rows.csv", index=False)
    f1_col = "F1 Score" if "F1 Score" in lazy_models_df.columns else "F1"
    print(f"LazyPredict benchmark rows: {len(lazy_models_df)}")
    display(lazy_models_df[["lazy_model_name", f1_col, "Balanced Accuracy", "Time Taken"]].head(15))



## 7) Selection of Top 3 Eligible Models

Rule enforced: only the top 3 **eligible** families from LazyPredict proceed to manual engineering.


In [ ]:

selected_top3 = select_top_eligible_from_lazypredict(lazy_models_df, top_n=3)

if len(selected_top3) < 3:
    fallback = pd.DataFrame(
        {
            "lazy_model_name": ["Fallback::LogisticRegression", "Fallback::LinearSVC", "Fallback::RandomForestClassifier"],
            "family_name": ["logistic_regression", "linear_svc", "random_forest"],
            "selection_reason": [
                "LazyPredict unavailable or insufficient eligible models.",
                "LazyPredict unavailable or insufficient eligible models.",
                "LazyPredict unavailable or insufficient eligible models.",
            ],
            "F1 Score": [np.nan, np.nan, np.nan],
        }
    )
    selected_top3 = fallback

selected_top3.to_csv(ARTIFACTS / "selected_top3_eligible_models.csv", index=False)
selected_families = selected_top3["family_name"].tolist()

print("Selected families for manual lab:", selected_families)
display(selected_top3)



## 8) Manual Engineering Lab

This lab manually implements only the selected top-3 families. It includes:

- train + CV + holdout evaluation,
- optional calibration for non-probabilistic models,
- threshold optimization for priority issue triage,
- qualitative error analysis.


In [ ]:

manual_rows = []
manual_artifacts = {}

for family in selected_families:
    row, artifacts = evaluate_manual_family(
        family_name=family,
        x_train=X_train_sparse,
        y_train=y_train,
        x_valid=X_valid_sparse,
        y_valid=y_valid,
        project_name=PROJECT_NAME,
        cv=4,
        random_state=RANDOM_STATE,
        calibrate_when_needed=True,
    )
    manual_rows.append(row)
    manual_artifacts[family] = artifacts

manual_results = pd.DataFrame(manual_rows).sort_values("holdout_primary_metric", ascending=False)
manual_results.to_csv(ARTIFACTS / "manual_engineering_results.csv", index=False)

display(
    manual_results[
        [
            "model_name",
            "cv_metric_mean",
            "holdout_primary_metric",
            "holdout_secondary_metric",
            "holdout_tertiary_metric",
            "calibration_metric",
            "train_time_sec",
        ]
    ]
)

family_from_display = {v: k for k, v in MANUAL_FAMILY_DISPLAY.items()}
best_manual_model_name = manual_results.iloc[0]["model_name"]
best_manual_family = family_from_display.get(best_manual_model_name)
best_manual = manual_artifacts[best_manual_family]

if best_manual["y_prob"] is not None and best_manual.get("classes_") is not None:
    threshold_result = optimize_priority_threshold(
        y_true=y_valid,
        y_prob=best_manual["y_prob"],
        class_order=best_manual["classes_"],
        priority_labels=("quality_issue", "shipping_service_issue"),
        beta=2.0,
    )
else:
    threshold_result = {
        "best_threshold": np.nan,
        "best_fbeta": np.nan,
        "priority_recall": np.nan,
        "priority_precision": np.nan,
    }

confidence = (
    best_manual["y_prob"].max(axis=1)
    if best_manual["y_prob"] is not None
    else np.full(len(y_valid), np.nan)
)
top20_review = qualitative_top20_review(
    texts=valid_df["clean_text"].tolist(),
    y_true=y_valid.tolist(),
    y_pred=best_manual["y_pred"].tolist(),
    confidences=confidence.tolist(),
    severity=valid_df["severity"].tolist(),
    high_priority_labels=("quality_issue", "shipping_service_issue"),
    top_n=20,
)
top20_review.to_csv(ARTIFACTS / "top20_prediction_review.csv", index=False)

print("Manual threshold optimization for priority issues:")
print(threshold_result)
display(top20_review)



## 9) FLAML Optimization Lab

FLAML is run as a full optimization track using macro F1 and CV-based search.


In [ ]:

flaml_row, flaml_details, flaml_model = run_flaml_optimization(
    x_train=X_train_sparse,
    y_train=y_train,
    x_valid=X_valid_sparse,
    y_valid=y_valid,
    project_name=PROJECT_NAME,
    time_budget=180,
    random_state=RANDOM_STATE,
    estimator_list=("lgbm", "xgboost", "rf", "extra_tree", "xgb_limitdepth"),
)

flaml_result_df = pd.DataFrame([flaml_row])
display(
    flaml_result_df[
        [
            "model_name",
            "cv_metric_mean",
            "holdout_primary_metric",
            "holdout_secondary_metric",
            "holdout_tertiary_metric",
            "train_time_sec",
            "interpretability_note",
        ]
    ]
)

with open(ARTIFACTS / "flaml_best_config.json", "w", encoding="utf-8") as f:
    json.dump(flaml_details, f, indent=2, default=str)

print("FLAML search summary")
print("- Best estimator:", flaml_details.get("best_estimator"))
print("- Best config:", flaml_details.get("best_config"))
print("- Best loss:", flaml_details.get("best_loss"))

best_manual_f1 = float(manual_results.iloc[0]["holdout_primary_metric"])
flaml_f1 = float(flaml_row.get("holdout_primary_metric", np.nan))
if pd.notna(flaml_f1):
    print(f"Macro F1 delta vs best manual: {flaml_f1 - best_manual_f1:+.4f}")



## 10) PyCaret Experiment Lab

PyCaret is treated as a full orchestration track:

- setup
- compare_models
- tune_model
- calibrate_model (where possible)
- finalize_model
- save_model


In [ ]:

pycaret_row, pycaret_details, pycaret_model = run_pycaret_experiment(
    x_train_dense=X_train_dense,
    y_train=y_train,
    x_valid_dense=X_valid_dense,
    y_valid=y_valid,
    feature_names=dense_feature_names,
    project_name=PROJECT_NAME,
    artifacts_dir=ARTIFACTS,
    session_id=RANDOM_STATE,
    fold=3,
    compare_top_n=3,
    tune_iter=20,
    include_models=("lr", "ridge", "rf", "et", "nb", "lda"),
)

pycaret_result_df = pd.DataFrame([pycaret_row])
display(
    pycaret_result_df[
        [
            "model_name",
            "cv_metric_mean",
            "holdout_primary_metric",
            "holdout_secondary_metric",
            "holdout_tertiary_metric",
            "calibration_metric",
            "interpretability_note",
        ]
    ]
)

if isinstance(pycaret_details.get("compare_table"), pd.DataFrame):
    print("PyCaret compare_models top rows:")
    display(pycaret_details["compare_table"].head(8))
if isinstance(pycaret_details.get("tune_table"), pd.DataFrame):
    print("PyCaret tune_model summary:")
    display(pycaret_details["tune_table"].head(5))

print("PyCaret saved model path:", pycaret_details.get("saved_model_path"))



## 11) Unified Leaderboard and Final Model Ranking

Unified comparison includes:

- top LazyPredict discovery rows,
- manual top-3 implementation rows,
- best FLAML result,
- best PyCaret finalized result,
- explicit baseline.


In [ ]:

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
t0 = pd.Timestamp.utcnow().timestamp()
baseline.fit(X_train_sparse, y_train)
y_pred_base = baseline.predict(X_valid_sparse)
y_prob_base = baseline.predict_proba(X_valid_sparse) if hasattr(baseline, "predict_proba") else None
baseline_train_time = pd.Timestamp.utcnow().timestamp() - t0

baseline_metrics = classification_metrics_bundle(
    y_true=y_valid,
    y_pred=y_pred_base,
    y_prob=y_prob_base,
    class_order=getattr(baseline, "classes_", None),
    high_priority_labels=("quality_issue", "shipping_service_issue"),
)

baseline_row = make_leaderboard_row(
    project_name=PROJECT_NAME,
    task_type="classification",
    library_source="baseline",
    model_name="DummyClassifier::most_frequent",
    cv_metric_mean=np.nan,
    cv_metric_std=np.nan,
    holdout_primary_metric=baseline_metrics["holdout_primary_metric"],
    holdout_secondary_metric=baseline_metrics["holdout_secondary_metric"],
    holdout_tertiary_metric=baseline_metrics["holdout_tertiary_metric"],
    calibration_metric=baseline_metrics["calibration_metric"],
    train_time_sec=baseline_train_time,
    infer_latency_ms=np.nan,
    model_size_mb=np.nan,
    interpretability_note="Operational sanity baseline.",
)

lazy_for_unified = lazy_rows_df.head(8).copy()

leaderboard_raw = pd.concat(
    [
        pd.DataFrame([baseline_row]),
        lazy_for_unified,
        pd.DataFrame(manual_rows),
        pd.DataFrame([flaml_row]),
        pd.DataFrame([pycaret_row]),
    ],
    ignore_index=True,
)

leaderboard_ranked = rank_leaderboard(leaderboard_raw)
leaderboard_ranked = save_leaderboard(leaderboard_ranked, ARTIFACTS / "leaderboard_nlp_unified.csv")
save_leaderboard(leaderboard_ranked, ARTIFACTS / "leaderboard_nlp_classification.csv")

print("Unified leaderboard top 10")
display(
    leaderboard_ranked[
        [
            "final_rank",
            "library_source",
            "model_name",
            "holdout_primary_metric",
            "holdout_secondary_metric",
            "holdout_tertiary_metric",
            "calibration_metric",
            "rank_score",
        ]
    ].head(10)
)

# Multi-seed rerun for top candidate families from the selected manual shortlist
seed_check_df = rerun_candidates_with_multiple_seeds(
    texts=model_df["clean_text"],
    labels=model_df[TARGET_COL],
    family_names=selected_families,
    seeds=[13, 42, 77],
    project_name=PROJECT_NAME,
    max_features=9000,
    ngram_range=(1, 2),
    test_size=0.2,
)
seed_check_df.to_csv(ARTIFACTS / "top3_candidates_multiseed.csv", index=False)

seed_summary = (
    seed_check_df.assign(base_model=seed_check_df["model_name"].str.replace(r"::seed_\d+", "", regex=True))
    .groupby("base_model", as_index=False)
    .agg(
        macro_f1_mean=("holdout_primary_metric", "mean"),
        macro_f1_std=("holdout_primary_metric", "std"),
        weighted_f1_mean=("holdout_secondary_metric", "mean"),
        critical_recall_mean=("holdout_tertiary_metric", "mean"),
    )
    .sort_values("macro_f1_mean", ascending=False)
)

seed_summary.to_csv(ARTIFACTS / "top3_candidates_multiseed_summary.csv", index=False)
print("Top-3 candidate stability across seeds")
display(seed_summary)

deployable = leaderboard_ranked[~leaderboard_ranked["library_source"].isin(["lazypredict"])].reset_index(drop=True)
winner_row = deployable.iloc[0]
runner_up_row = deployable.iloc[1] if len(deployable) > 1 else deployable.iloc[0]

winner_margin = float(winner_row["holdout_primary_metric"] - runner_up_row["holdout_primary_metric"]) if pd.notna(winner_row["holdout_primary_metric"]) and pd.notna(runner_up_row["holdout_primary_metric"]) else np.nan

rationale = (
    f"Winner by unified ranking: {winner_row['model_name']} ({winner_row['library_source']}) with "
    f"macro F1={winner_row['holdout_primary_metric']:.4f}, weighted F1={winner_row['holdout_secondary_metric']:.4f}, "
    f"critical-issue recall={winner_row['holdout_tertiary_metric']:.4f}. "
    f"Second-best candidate: {runner_up_row['model_name']} with macro F1={runner_up_row['holdout_primary_metric']:.4f}. "
    f"Margin on macro F1: {winner_margin:+.4f}."
)

(ARTIFACTS / "final_model_selection_rationale.txt").write_text(rationale, encoding="utf-8")
print(rationale)



## 12) Business Recommendation

Convert modeling outcomes into concrete operating decisions.


In [ ]:

issue_volume = model_df[TARGET_COL].value_counts().rename_axis("issue_type").reset_index(name="volume")
error_mix = top20_review["error_tag"].value_counts().rename_axis("error_tag").reset_index(name="count")

recommendation_df = pd.DataFrame(
    [
        {
            "priority": 1,
            "recommendation": "Route high priority issue predictions to daily support triage queue.",
            "evidence": f"Critical issue recall focus metric tracked in leaderboard. Top class volume: {issue_volume.iloc[0]['issue_type']}",
        },
        {
            "priority": 2,
            "recommendation": "Use weekly topic drill-down on top categories for product/ops owners.",
            "evidence": "Category-level frequency and qualitative error tags expose recurring pain points.",
        },
        {
            "priority": 3,
            "recommendation": "Add human QA review for overconfident errors before customer-impacting automation.",
            "evidence": f"Top error tag counts include: {error_mix.head(3).to_dict(orient='records')}",
        },
    ]
)

recommendation_df.to_csv(ARTIFACTS / "business_recommendations.csv", index=False)

display(issue_volume.head(10))
display(error_mix)
display(recommendation_df)



## 13) Inference / Deployment Path

Deploy an inference bundle (vectorizer + model + metadata) and show test predictions.


In [ ]:

family_from_display = {v: k for k, v in MANUAL_FAMILY_DISPLAY.items()}

# Prefer winner if manual; otherwise pick the strongest manual candidate as safer operational fallback.
if winner_row["library_source"] == "manual_engineering":
    deployment_family = family_from_display.get(winner_row["model_name"])
elif runner_up_row["library_source"] == "manual_engineering":
    deployment_family = family_from_display.get(runner_up_row["model_name"])
else:
    deployment_family = selected_families[0]

deployment_model = manual_artifacts[deployment_family]["model"]
deployment_classes = getattr(deployment_model, "classes_", sorted(model_df[TARGET_COL].unique()))

bundle_metadata = {
    "project_name": PROJECT_NAME,
    "created_utc": datetime.utcnow().isoformat(),
    "training_rows": int(len(train_df)),
    "deployment_family": deployment_family,
    "winner_model_name": winner_row["model_name"],
    "winner_library": winner_row["library_source"],
    "priority_threshold": threshold_result.get("best_threshold"),
}

bundle_path = save_inference_bundle(
    ARTIFACTS / "inference_bundle.pkl",
    vectorizer=vectorizer,
    model=deployment_model,
    class_order=deployment_classes,
    priority_threshold=threshold_result.get("best_threshold"),
    metadata=bundle_metadata,
)

loaded_bundle = load_inference_bundle(bundle_path)
sample_texts = [
    "Package arrived late and the container was leaking.",
    "Love the taste, this is my favorite snack.",
    "Too expensive for such a small quantity.",
    "The product smells stale and seems expired.",
]
deploy_preview = predict_with_inference_bundle(loaded_bundle, sample_texts)
deploy_preview.to_csv(ARTIFACTS / "inference_preview.csv", index=False)

print(f"Saved inference bundle: {bundle_path}")
display(deploy_preview)



## 14) Monitoring / Drift / Retraining Plan

Monitoring must track both model quality and business usefulness.


In [ ]:

monitoring_plan = pd.DataFrame(
    [
        {
            "signal": "Macro F1 on rolling labeled holdout",
            "threshold": "Drop > 0.05 from current champion",
            "cadence": "Weekly",
            "action": "Trigger candidate retraining and challenger evaluation",
        },
        {
            "signal": "Critical issue recall",
            "threshold": "Below 0.70",
            "cadence": "Daily",
            "action": "Increase manual QA and threshold sensitivity",
        },
        {
            "signal": "Input text drift (token length, OOV rate)",
            "threshold": "KS/PSI alert or OOV > 20%",
            "cadence": "Weekly",
            "action": "Refresh vocabulary and retrain vectorizer + models",
        },
        {
            "signal": "Error-tag mix (missed_critical_issue)",
            "threshold": "Share above 15% in QA sample",
            "cadence": "Weekly",
            "action": "Root-cause review and relabeling sprint",
        },
    ]
)

drift_baseline = {
    "train_text_len_tokens_quantiles": train_df["text_len_tokens"].quantile([0.25, 0.5, 0.75]).to_dict(),
    "train_label_distribution": train_df[TARGET_COL].value_counts(normalize=True).to_dict(),
    "vocab_size": int(len(vectorizer.vocabulary_)),
}

monitoring_plan.to_csv(ARTIFACTS / "monitoring_plan.csv", index=False)
with open(ARTIFACTS / "drift_baseline.json", "w", encoding="utf-8") as f:
    json.dump(drift_baseline, f, indent=2)

display(monitoring_plan)
print("Saved drift baseline and monitoring plan.")



## 15) Limitations and Next Steps

**Current limitations**

- Labels are weakly supervised and partially keyword-derived.
- Single holdout split can still miss rare operational edge cases.
- PyCaret and FLAML behaviors can vary by dependency versions and compute budget.

**Next steps**

1. Add human-verified labels for at least 1,000 examples focused on critical classes.
2. Add retrieval + topic drill-down as a complementary diagnostic panel for class errors.
3. Add CI job to run reduced notebook smoke tests and schema checks on leaderboard outputs.
4. Promote champion/challenger registry with scheduled retraining and rollback guardrails.


In [ ]:

artifact_manifest = sorted([p.name for p in ARTIFACTS.glob("*")])
print("Artifacts currently in folder:")
for name in artifact_manifest:
    print("-", name)
